# TruePin

In [35]:
import pandas as pd
import geopandas as gpd
from shapely import wkb
from shapely.geometry import box

df = pd.read_parquet("project_d_samples.parquet")
df.head()

,id,geometry,bbox,type,version,sources,names,categories,confidence,websites,socials,emails,phones,brand,addresses
0,08f266694b8dd34103db5d8f268c8bfe,b'\x01\x01\x00\x00\x00@]\xee\xdd0\rV\xc0\xdb\x...,"{'xmax': -88.20610046386719, 'xmin': -88.20610...",place,0,"[{'confidence': 0.8770555990602976, 'dataset':...","{'common': None, 'primary': 'SNT Biotech Lab',...","{'alternate': ['health_and_medical', 'laborato...",0.877056,[http://www.sntbiotech.com/],[https://www.facebook.com/109093534897634],None,[+13127662326],None,"[{'country': 'US', 'freeform': '24119 W Riverw..."
1,08f2af608b8633ac03a9430e8ba43b40,b'\x01\x01\x00\x00\x00\\\xef\xedLW$S\xc0)lPE;\...,"{'xmax': -76.56782531738281, 'xmin': -76.56784...",place,0,"[{'confidence': 0.804932735426009, 'dataset': ...","{'common': None, 'primary': 'Debbie’s Doula Se...","{'alternate': ['prenatal_perinatal_care'], 'pr...",0.804933,[http://debbiesdoulaservices.com/],[https://www.facebook.com/100596505513704],None,[+17572685658],None,"[{'country': 'US', 'freeform': '912 Arnett Dr'..."
2,08f44a112bcc56d30324dd2659dd9b1b,b'\x01\x01\x00\x00\x00\xc9v\xbe\x9f\x1a\x0cT\x...,"{'xmax': -80.18911743164062, 'xmin': -80.18912...",place,0,"[{'confidence': 0.9793990828827596, 'dataset':...","{'common': None, 'primary': 'Wax Custom Commun...","{'alternate': ['business_advertising', 'market...",0.995262,[http://www.waxcom.com],[https://www.facebook.com/106355339205],None,[+13053505700],None,"[{'country': 'US', 'freeform': '261 NE 1st St'..."
3,08f44dab0c4b02db035754f1e3be0658,b'\x01\x01\x00\x00\x00\x00-\xd6\x90U/T\xc0\xd9...,"{'xmax': -80.73958587646484, 'xmin': -80.73960...",place,0,"[{'confidence': 0.77, 'dataset': 'Microsoft', ...","{'common': None, 'primary': 'Advance Auto Part...","{'alternate': ['automotive', 'retail'], 'prima...",0.770000,[https://stores.advanceautoparts.com/nc/charlo...,None,None,[7045664047],None,"[{'country': 'US', 'freeform': '3020 Milton Rd..."
4,08f26e0cce79a7b603caea464ec624bb,b'\x01\x01\x00\x00\x00\xe0\x80\x96\xae\xe0XX\x...,"{'xmax': -97.38871002197266, 'xmin': -97.38872...",place,0,"[{'confidence': 0.9793990828827596, 'dataset':...","{'common': None, 'primary': 'Tanglefoot Wine C...","{'alternate': None, 'primary': 'winery'}",0.979399,[http://Tanglefootwinecellar.com/],[https://www.facebook.com/101890712147590],None,[+17855772944],None,"[{'country': 'US', 'freeform': '9925 S Amos Rd..."


In [36]:
df["geometry"] = df["geometry"].apply(wkb.loads)
gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

gdf.geometry.type.value_counts()

Point    3425
Name: count, dtype: int64

In [37]:
gdf[["geometry", "bbox"]].head()

,geometry,bbox
0,POINT (-88.20611 41.61867),"{'xmax': -88.20610046386719, 'xmin': -88.20610..."
1,POINT (-76.56783 37.13462),"{'xmax': -76.56782531738281, 'xmin': -76.56784..."
2,POINT (-80.18912 25.77545),"{'xmax': -80.18911743164062, 'xmin': -80.18912..."
3,POINT (-80.7396 35.23658),"{'xmax': -80.73958587646484, 'xmin': -80.73960..."
4,POINT (-97.38871 38.656),"{'xmax': -97.38871002197266, 'xmin': -97.38872..."


In [38]:
def bbox_to_polygon(b):
    return box(b["xmin"], b["ymin"], b["xmax"], b["ymax"])

gdf["bbox_geom"] = gdf["bbox"].apply(bbox_to_polygon)

In [40]:
gdf["bbox_geom"] = gpd.GeoSeries(gdf["bbox_geom"], crs=gdf.crs)
gdf.geometry.within(gdf["bbox_geom"]).value_counts()

True     3416
False       9
Name: count, dtype: int64

In [41]:
gdf.loc[~gdf.geometry.within(gdf["bbox_geom"]), 
        ["geometry", "bbox", "categories", "confidence"]].head(10)


,geometry,bbox,categories,confidence
1005,POINT (-73.97113 40.89867),"{'xmax': -73.97113037109375, 'xmin': -73.97113...","{'alternate': None, 'primary': 'personal_injur...",0.77
1028,POINT (-106.63314 35.04987),"{'xmax': -106.63314056396484, 'xmin': -106.633...","{'alternate': ['hotel', 'inn', 'motel'], 'prim...",0.77
2123,POINT (-76.55529 38.78493),"{'xmax': -76.55529022216797, 'xmin': -76.55529...","{'alternate': ['fishing_charter', 'active_life...",0.77
2160,POINT (-80.14058 25.79691),"{'xmax': -80.14057922363281, 'xmin': -80.14057...","{'alternate': None, 'primary': 'public_relatio...",0.77
2204,POINT (-93.86113 32.44955),"{'xmax': -93.86112976074219, 'xmin': -93.86112...","{'alternate': ['hotel'], 'primary': 'resort'}",0.77
2337,POINT (-122.38413 40.59862),"{'xmax': -122.3841323852539, 'xmin': -122.3841...","{'alternate': ['hotel', 'inn', 'motel'], 'prim...",0.77
2669,POINT (-97.15608 33.38482),"{'xmax': -97.15608215332031, 'xmin': -97.15608...","{'alternate': None, 'primary': 'septic_services'}",0.77
2670,POINT (-96.97173 32.99805),"{'xmax': -96.97172546386719, 'xmin': -96.97174...","{'alternate': ['electrician', 'utility_service...",0.77
2807,POINT (-117.80508 44.78466),"{'xmax': -117.8050765991211, 'xmin': -117.8050...","{'alternate': ['hotel', 'inn', 'motel'], 'prim...",0.77
